# B. Code comparisons

This notebook follows Appendix B of Paper I and the comparisons of Secs. 2.2 and 2.3.
Every code is compared with every other, one parameter varied at a time about `PLANCK18` over a five-point grid: the background $E(z)$ and $\chi(z)$, their derivatives, and the linear $P_{\rm m}$ and $P_{\rm cb}$ with their derivatives.
We recompute one row against CAMB if it is installed, then rebuild the four tables of the appendix from `accuracy.json`, which holds the results of these comparisons.

In [1]:
import jax
jax.config.update("jax_enable_x64", True)   # before the first ggah_mod import

from importlib.metadata import version

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import _helpers as T
T.paper_style()
FAST = T.fast_mode()
for name in ("ggah_mod", "emu_pk", "jax", "numpy"):
    print(f"{name:9s} {version(name)}")
print("FAST =", FAST)

import json
import re

import pandas as pd

from ggah_mod.cosmology import PLANCK18, comoving_distance, hubble_e

A = json.load(open(T.data_path("comparisons/accuracy.json")))
print(len(A), "measurements")

ggah_mod  1.7.0
emu_pk    2.1.1
jax       0.10.2
numpy     2.4.6
FAST = False


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


2664 measurements


## One row, recomputed

For each pair of codes and each parameter, the cell is the largest fractional deviation over the five-point grid and over $z\in[0.01, 3]$, for $E$ and for $\chi$ separately.
We recompute this package against CAMB for the $\Omega_{\rm m}$ scan.
`camb_input` translates a `Cosmology` into CAMB's parameters with the same neutrino convention.

In [2]:
z = np.linspace(0.01, 3.0, 60)
GRID = {"Omega_m": (0.24, 0.28, 0.31, 0.35, 0.40), "w0": (-1.3, -1.15, -1.0, -0.85, -0.7)}
try:
    import camb
    from ggah_mod.cosmology.power import camb_input
    for p, grid in GRID.items():
        de = dx = 0.0
        for v in grid:
            c = PLANCK18.replace(**{p: v})
            bg = camb.get_background(camb_input(c))
            e_c = bg.hubble_parameter(z) / bg.hubble_parameter(0.0)
            x_c = bg.comoving_radial_distance(z) * c.h
            de = max(de, float(np.max(np.abs(np.asarray(hubble_e(z, c)) / e_c - 1))))
            dx = max(dx, float(np.max(np.abs(np.asarray(comoving_distance(z, c)) / x_c - 1))))
        print(f"{p:8s} E: {de:.1e} (paper {A[f'scan {p}: E, ggah vs CAMB']:.1e})   "
              f"chi: {dx:.1e} (paper {A[f'scan {p}: chi, ggah vs CAMB']:.1e})")
except ImportError:
    print("camb is not installed (pip install camb): skipped")

Omega_m  E: 2.6e-08 (paper 2.6e-08)   chi: 1.3e-08 (paper 1.3e-08)


w0       E: 2.1e-08 (paper 2.1e-08)   chi: 1.0e-08 (paper 1.0e-08)


## The four tables

A cell holds two numbers, $E$/$\chi$ for the background and $P_{\rm m}$/$P_{\rm cb}$ for the power spectrum; a dash is a parameter one of the pair does not take.
The derivative tables start with a tier of self-checks, each differentiable code against its own finite differences.

In [3]:
def fmt(x):
    return "-" if x is None or x != x else f"{x:.1e}"


def table(prefix, params, pairs, q):
    """Rows: pairs; columns: params; cell 'q0 / q1' from keys '<prefix> <p>: <q>, <pair>'."""
    rows = {}
    for pair in pairs:
        rows[pair] = [fmt(A.get(f"{prefix} {p}: {q[0]}, {pair}")) + " / " + fmt(A.get(f"{prefix} {p}: {q[1]}, {pair}"))
                      for p in params]
    return pd.DataFrame.from_dict(rows, orient="index", columns=params)


def pairs_of(prefix, q):
    seen = []
    for k in A:
        m = re.match(rf"{re.escape(prefix)} \S+: {q}, (.+)$", k)
        if m and m.group(1) not in seen:
            seen.append(m.group(1))
    return seen


BG = ["Omega_m", "Omega_b", "h", "sum_mnu", "w0", "wa", "Omega_k", "T_cmb"]
PK = ["Omega_m", "Omega_b", "h", "n_s", "ln10A_s", "sum_mnu", "w0", "wa", "Omega_k", "T_cmb"]
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
print("Table B.1: background values, E / chi")
print(table("scan", BG, pairs_of("scan", "E"), ("E", "chi")).to_string())

Table B.1: background values, E / chi
                                Omega_m            Omega_b                  h            sum_mnu                 w0                 wa            Omega_k              T_cmb
CAMB vs CLASS         2.9e-08 / 1.4e-08  2.3e-08 / 1.1e-08  2.9e-08 / 1.4e-08  2.3e-08 / 1.1e-08  2.3e-08 / 1.3e-08  9.9e-08 / 1.3e-07  2.4e-08 / 1.2e-08  2.3e-08 / 1.1e-08
CAMB vs jax_cosmo     3.2e-03 / 1.4e-02  2.5e-03 / 1.3e-02  3.2e-03 / 1.3e-02              - / -  2.6e-03 / 1.5e-02  2.6e-03 / 1.3e-02  2.6e-03 / 1.3e-02              - / -
CCL vs CAMB           1.2e-06 / 5.2e-07  1.2e-06 / 4.3e-07  1.2e-06 / 4.3e-07  1.2e-06 / 4.3e-07  1.2e-06 / 5.2e-07  1.2e-06 / 5.2e-07  1.2e-06 / 4.6e-07  1.2e-06 / 4.3e-07
CCL vs CLASS          1.2e-06 / 5.3e-07  1.1e-06 / 4.3e-07  1.1e-06 / 4.3e-07  1.1e-06 / 4.3e-07  1.2e-06 / 5.3e-07  1.3e-06 / 6.5e-07  1.2e-06 / 4.6e-07  1.1e-06 / 4.3e-07
CCL vs jax_cosmo      3.2e-03 / 1.4e-02  2.5e-03 / 1.3e-02  3.2e-03 / 1.3e-02              - / - 

In [4]:
print("Table B.2: background derivatives, E / chi")
print(table("dscan", BG, pairs_of("dscan", "E"), ("E", "chi")).to_string())

Table B.2: background derivatives, E / chi
                                Omega_m            Omega_b                  h            sum_mnu                 w0                 wa            Omega_k              T_cmb
CAMB vs CLASS         4.6e-08 / 1.3e-07  4.4e-13 / 7.5e-12  6.5e-05 / 3.8e-04  5.8e-03 / 4.5e-03  3.5e-08 / 2.4e-07  4.2e-08 / 9.4e-07  4.0e-08 / 1.5e-06  4.9e-05 / 3.0e-03
CAMB vs jax_cosmo     5.0e-03 / 1.4e-02  2.2e-13 / 0.0e+00  1.2e+00 / 1.1e+00              - / -  5.0e-03 / 4.5e-02  5.9e-03 / 5.7e-03  4.1e-03 / 2.4e-02              - / -
CCL vs CAMB           9.2e-08 / 1.3e-06  2.2e-13 / 0.0e+00  6.1e-05 / 7.1e-05  5.8e-03 / 4.6e-03  3.4e-07 / 1.6e-06  6.7e-07 / 3.7e-06  7.7e-07 / 1.7e-06  1.4e-04 / 1.0e-04
CCL vs CLASS          6.2e-08 / 1.4e-06  4.4e-13 / 7.5e-12  9.0e-06 / 3.8e-04  2.0e-04 / 2.7e-04  3.3e-07 / 1.9e-06  6.5e-07 / 2.8e-06  7.4e-07 / 3.2e-06  9.7e-05 / 3.1e-03
CCL vs jax_cosmo      5.0e-03 / 1.4e-02  0.0e+00 / 0.0e+00  1.2e+00 / 1.1e+00              -

In [5]:
for prefix, title in (("pkscan", "Table B.3: P(k) values at the fiducial, Pm / Pcb"),
                      ("pkscan-base", "Table B.3, lower tier: P(k) values at the common baseline, Pm / Pcb"),
                      ("pkdscan", "Table B.4: P(k) derivatives at the fiducial, Pm / Pcb"),
                      ("pkdscan-base", "Table B.4, lower tier: P(k) derivatives at the baseline, Pm / Pcb")):
    print(title)
    print(table(prefix, PK, pairs_of(prefix, "Pm"), ("Pm", "Pcb")).to_string())
    print()

Table B.3: P(k) values at the fiducial, Pm / Pcb
                              Omega_m            Omega_b                  h                n_s            ln10A_s            sum_mnu                 w0                 wa            Omega_k              T_cmb
CAMB vs SymBoltz    8.6e-03 / 8.5e-03  7.9e-03 / 7.8e-03  8.0e-03 / 7.9e-03  7.8e-03 / 7.7e-03  7.8e-03 / 7.7e-03  1.1e-02 / 1.0e-02              - / -              - / -              - / -              - / -
CAMB vs emu_pk      7.8e-04 / 8.3e-04  9.7e-04 / 9.0e-04  9.6e-04 / 9.1e-04  6.4e-04 / 5.9e-04  6.4e-04 / 5.9e-04  9.0e-04 / 1.0e-03  7.2e-04 / 6.4e-04  8.4e-04 / 7.9e-04  7.1e-04 / 6.1e-04              - / -
CLASS vs CAMB       9.0e-04 / 9.9e-04  7.4e-04 / 6.3e-04  8.4e-04 / 9.3e-04  6.1e-04 / 6.1e-04  6.1e-04 / 6.1e-04  5.7e-03 / 4.6e-03  6.1e-04 / 6.2e-04  6.1e-04 / 6.1e-04  6.5e-04 / 6.1e-04  6.2e-04 / 6.4e-04
CLASS vs SymBoltz   8.7e-03 / 8.5e-03  8.1e-03 / 8.0e-03  8.2e-03 / 8.0e-03  8.0e-03 / 7.9e-03  8.0e-03 / 7.9e-03  

## Exercises

1. Install `astropy` and add it to the recomputed row through `astropy.cosmology.w0waCDM`. Why does it sit $10^{-5}$ from the others in $\sum m_\nu$ (Sec. 2.1, the neutrino convention)?
2. Recompute the finite-difference self-check of this package (the `AD vs FD` rows) for $\Omega_{\rm m}$ with a step of $10^{-4}$, and then $10^{-6}$. Which step does the table use, and why does the smaller one not do better?
3. Pick the worst cell of Table B.3 that involves `emu_pk` and explain it from Sec. 2.3.